In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import nltk

nltk.download('punkt')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving spam.csv to spam (1).csv


In [ ]:
df = pd.read_csv("spam.csv", encoding='latin-1')

In [ ]:
df.head()

In [ ]:
df.shape
df.columns

In [ ]:
df = df[['v1','v2']]

In [ ]:
df.rename(columns={
    'v1':'target',
    'v2':'text'
}, inplace=True)

In [ ]:
df.isnull().sum()
df.duplicated().sum()
df = df.drop_duplicates(keep='first')

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df['target'] = encoder.fit_transform(df['target'])

In [ ]:
df['target'].value_counts()
plt.pie(
    df['target'].value_counts(),
    labels=['Ham','Spam'],
    autopct='%0.2f%%'
)

plt.show()
df['num_characters'] = df['text'].apply(len)
df['num_words'] = df['text'].apply(
    lambda x: len(x.split())
)


df['num_sentences'] = df['text'].apply(
    lambda x: len(nltk.sent_tokenize(x))
)
df.describe()
df[df['target']==0].describe()
df[df['target']==1].describe()

In [ ]:
sns.histplot(
    df[df['target']==0]['num_characters']
)

sns.histplot(
    df[df['target']==1]['num_characters']
)

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

In [ ]:
print(df.columns)

In [ ]:
stop_words = set(stopwords.words('english'))

In [ ]:

import string

def transform_text(text):

    text = text.lower()

    text = nltk.word_tokenize(text)

    y = []

    for i in text:
        if i.isalnum():
            y.append(i)

    text = y.copy()
    y.clear()

    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)

    text = y.copy()
    y.clear()

    for i in text:
        y.append(ps.stem(i))

    return " ".join(y)


In [ ]:
df['transformed_text'] = df['text'].apply(transform_text)

In [ ]:
spam_words = []

for msg in df[df['target']==1]['transformed_text']:
    for word in msg.split():
        spam_words.append(word)

print(len(spam_words))

In [ ]:
from collections import Counter

Counter(spam_words).most_common(20)

In [ ]:
sns.barplot(
    x=pd.DataFrame(
        Counter(spam_words).most_common(20)
    )[0],
    y=pd.DataFrame(
        Counter(spam_words).most_common(20)
    )[1]
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
y = df['target'].values

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,y_train,y_test = \
train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=2
)

In [ ]:
from sklearn.naive_bayes import MultinomialNB

mnb = MultinomialNB()

In [ ]:
mnb.fit(X_train,y_train)

In [ ]:
y_pred = mnb.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
print(
    accuracy_score(y_test,y_pred)
)
print(
    precision_score(y_test,y_pred)
)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score

In [ ]:
models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'Decision Tree': DecisionTreeClassifier(),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(kernel='linear')
}

In [ ]:
results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(y_test, y_pred)

    results.append([
        name,
        accuracy,
        precision
    ])

In [ ]:
results_df = pd.DataFrame(
    results,
    columns=[
        'Model',
        'Accuracy',
        'Precision'
    ]
)

In [ ]:
results_df.sort_values(
    by='Precision',
    ascending=False,
    inplace=True
)

In [ ]:
results_df

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.bar(
    results_df['Model'],
    results_df['Accuracy']
)

plt.xticks(rotation=45)

plt.ylabel("Accuracy")

plt.title("Model Comparison")

plt.show()

In [ ]:
results_df.plot(
    x='Model',
    y=['Accuracy','Precision'],
    kind='bar',
    figsize=(10,5)
)

plt.title("Accuracy vs Precision")

plt.show()

In [ ]:
DecisionTreeClassifier(random_state=42)

RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

LogisticRegression(
    max_iter=1000,
    random_state=42
)

In [ ]:
sns.countplot(
    x=df['target']
)
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred
    )
)

In [ ]:
print(results_df)

In [ ]:

from sklearn.naive_bayes import MultinomialNB

best_model = MultinomialNB()

best_model.fit(X_train, y_train)

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X = tfidf.fit_transform(
    df['transformed_text']
).toarray()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=2,
    stratify=y
)


In [ ]:
from sklearn.naive_bayes import MultinomialNB

best_model = MultinomialNB()

best_model.fit(
    X_train,
    y_train
)

MultinomialNB()

In [ ]:
print(best_model.n_features_in_)
print(len(tfidf.vocabulary_))

5000
5000


In [ ]:
import pickle

pickle.dump(best_model, open('spam_model.pkl', 'wb'))
pickle.dump(tfidf, open('vectorizer.pkl', 'wb'))
pickle.dump(encoder, open('label_encoder.pkl', 'wb'))

In [ ]:
from google.colab import files

files.download('spam_model.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download('vectorizer.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download('label_encoder.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from sklearn.metrics import confusion_matrix
y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

print(classification_report(y_test, y_pred))

In [ ]:
sns.heatmap(
    cm,
    annot=True,
    fmt='d'
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:

import pickle

pickle.dump(tfidf, open('vectorizer.pkl','wb'))
pickle.dump(best_model, open('spam_model.pkl','wb'))
pickle.dump(encoder, open('label_encoder.pkl','wb'))

In [ ]:
!ls

 label_encoder.pkl  'spam (1).csv'   spam_model.pkl
 sample_data	     spam.csv	     vectorizer.pkl


In [ ]:
from google.colab import files

files.download('spam_model.pkl')
files.download('vectorizer.pkl')
files.download('label_encoder.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(tfidf)

TfidfVectorizer(max_features=5000)


In [ ]:
print(encoder)

LabelEncoder()


In [ ]:
print(tfidf)
print(encoder)

TfidfVectorizer(max_features=5000)
LabelEncoder()


In [ ]:
files.download('vectorizer.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download('label_encoder.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pickle

pickle.dump(tfidf, open('vectorizer.pkl', 'wb'))
pickle.dump(encoder, open('label_encoder.pkl', 'wb'))

In [ ]:
import pickle

pickle.dump(tfidf, open('vectorizer.pkl', 'wb'))
pickle.dump(encoder, open('label_encoder.pkl', 'wb'))

In [ ]:
import os
os.listdir()

['.config',
 'label_encoder.pkl',
 'vectorizer.pkl',
 'spam_model.pkl',
 'spam.csv',
 'spam (1).csv',
 'sample_data']

In [ ]:
pickle.dump(
    tfidf,
    open('vectorizer.pkl','wb')
)

In [ ]:
pickle.dump(
    best_model,
    open('spam_model.pkl','wb')
)

In [ ]:
def predict_spam(message):

    transformed = transform_text(message)

    vector = tfidf.transform(
        [transformed]
    )

    prediction = best_model.predict(
        vector
    )[0]

    if prediction == 1:
        return "Spam"

    return "Ham"

In [ ]:
predict_spam(
    "Congratulations! You won a free iPhone"
)

In [ ]:
messages = [
    "Hey where are you?",
    "You have won $1000",
    "Let's meet tomorrow",
    "Claim your free prize now"
]

for msg in messages:
    print(
        msg,
        "->",
        predict_spam(msg)
    )

In [ ]:
from scipy.sparse import hstack

X_text = tfidf.fit_transform(
    df['transformed_text']
)

X_extra = df[
[
'num_characters',
'num_words',
'num_sentences'
]
].values

X = hstack([
    X_text,
    X_extra
])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=2,
    stratify=y
)

In [ ]:
best_model = MultinomialNB()
best_model.fit(X_train, y_train)

In [ ]:
y_pred = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))

In [ ]:
from sklearn.metrics import recall_score, f1_score

recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

In [ ]:
results.append([
    name,
    accuracy,
    precision,
    recall,
    f1
])

In [ ]:
print(tfidf)
print(best_model)

TfidfVectorizer(max_features=5000)
MultinomialNB()
